# Chapter 1: Introduction to Language Models

> **Goal:** Understand what a language model is, why it matters, and run your first LLM in code.

---

## 🧠 The Big Idea

A **Language Model (LM)** is a system that learns the statistical patterns of language so it can predict what word (or token) comes next in a sequence.

That's it. Everything — ChatGPT, Claude, Gemini — is built on this one simple idea: **predict the next token**.

```
Input:  "The cat sat on the"
Output: "mat" (most likely next word)
```

But when you train this on trillions of words, magic happens — the model learns grammar, facts, reasoning, coding, and much more.

---

## 📖 A Brief History

| Era | Model Type | Key Idea | Limitation |
|-----|-----------|----------|------------|
| 1950s-2000s | N-gram | Count word co-occurrences | Forgets long-range context |
| 2013-2017 | RNN / LSTM | Recurrent hidden state | Sequential (slow), vanishing gradients |
| 2017 | Transformer | Attention mechanism | Needs huge compute |
| 2018+ | BERT, GPT | Pretrain on large text | Fine-tune for specific tasks |
| 2022+ | ChatGPT, LLaMA | RLHF alignment | Hallucinations, cost |

---

## 🔄 How a Language Model Learns

The training process is called **self-supervised learning** — no human labels needed! The model's job is to predict the hidden word:

```
Training sentence: "Cats like to drink [MASK] in the morning"
Target:             "milk"

Training sentence: "She opened the door and [MASK] inside"
Target:             "walked"
```

The model sees billions of these examples and adjusts its internal weights to get better and better at prediction.

---

## 💻 Let's Run Our First Language Model

In [ ]:
# First, install the required library (uncomment if not installed)
# !pip install transformers torch

from transformers import pipeline

# Create a text generation pipeline using a small open model
# This downloads ~500MB on first run — it's cached after that
generator = pipeline(
    'text-generation',
    model='distilgpt2',  # A small, fast version of GPT-2
    device=-1            # -1 = CPU, 0 = first GPU
)

# Generate text from a prompt
prompt = "The future of artificial intelligence is"

result = generator(
    prompt,
    max_new_tokens=50,   # Generate up to 50 new tokens
    num_return_sequences=3,  # Generate 3 different completions
    temperature=0.8,     # 0.0 = deterministic, 1.0 = very creative
    do_sample=True       # Use sampling (randomness)
)

print(f"Prompt: {prompt}\n")
for i, r in enumerate(result):
    print(f"--- Completion {i+1} ---")
    print(r['generated_text'])
    print()

### What just happened?

1. We loaded a **pre-trained model** (`distilgpt2`) — it already knows English from its training data.
2. We gave it a **prompt** (a starting sentence).
3. The model **sampled** from its probability distribution to generate the next tokens one by one.
4. With `temperature=0.8`, it injects some creativity — the same prompt gives different results each time.

---

## 🎲 Understanding Temperature

**Temperature** controls how creative (or boring) the model is:

- `temperature=0.0` → Always picks the single most likely word (deterministic)
- `temperature=1.0` → Picks from the full probability distribution (creative but risky)
- `temperature=0.7` → Good balance (recommended for most tasks)

Think of it like a **spice dial**. Too low = bland. Too high = chaotic.

In [ ]:
# Demonstrate the effect of temperature
prompt = "Once upon a time in a faraway land"

for temp in [0.1, 0.7, 1.5]:
    result = generator(
        prompt,
        max_new_tokens=30,
        temperature=max(temp, 0.01),  # temperature cannot be 0 with do_sample=True
        do_sample=True
    )
    print(f"Temperature={temp}:")
    print(result[0]['generated_text'])
    print()

## 🔍 What Makes an LLM "Large"?

The "L" in LLM stands for **Large**. But what does that mean in practice?

| Model | Parameters | VRAM Needed | What It Can Do |
|-------|-----------|-------------|----------------|
| DistilGPT-2 | 82 million | < 1GB | Basic text generation |
| GPT-2 (full) | 1.5 billion | ~3GB | Decent text generation |
| LLaMA 3 8B | 8 billion | ~6GB | Instruction following, reasoning |
| Claude 3.5 | ~100B+ | Cloud only | Near-human performance |
| GPT-4 | ~1 trillion | Cloud only | Complex reasoning, coding |

**Parameters** are the numbers the model adjusts during training. More parameters = more capacity to memorize patterns = better performance (up to a point).

---

## 🏋️ The Training Process Visualized

```
1. START with random weights (the model knows nothing)
   ↓
2. SHOW it a text: "The sky is [MASK]"
   ↓
3. MODEL PREDICTS: "blue" (maybe wrong at first)
   ↓
4. COMPARE with true answer: "blue" (correct!) or "green" (wrong!)
   ↓
5. ADJUST weights slightly to reduce the error
   ↓
6. REPEAT for billions of examples
   ↓
7. RESULT: a model that "understands" language patterns
```

This process uses an algorithm called **gradient descent** — the model slowly "walks downhill" toward better predictions.

---

## 🧪 Hands-On Exercise: Explore Model Confidence

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer manually for more control
model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()  # Set to evaluation mode (disables dropout)

# Encode the prompt into token IDs
prompt = "The capital of France is"
input_ids = tokenizer.encode(prompt, return_tensors='pt')

# Run the model and get the probability distribution for the NEXT token
with torch.no_grad():  # No gradient tracking needed for inference
    outputs = model(input_ids)
    next_token_logits = outputs.logits[0, -1, :]  # Last token's predictions

# Convert raw logits to probabilities using softmax
probs = torch.softmax(next_token_logits, dim=-1)

# Show top 10 candidate next tokens
top_k = torch.topk(probs, 10)

print(f"Prompt: '{prompt}'")
print("\nTop 10 predicted next tokens:")
print(f"{'Token':<20} {'Probability':<15}")
print("-" * 35)
for prob, token_id in zip(top_k.values, top_k.indices):
    token_text = tokenizer.decode([token_id.item()])
    print(f"{repr(token_text):<20} {prob.item():.4f}")

### Key Insight

The model does NOT output a single word — it outputs a **probability distribution over its entire vocabulary** (50,000+ words).

Token generation is then a **sampling** process: pick the next token based on these probabilities.

This is why you get different outputs each time you run the same prompt with `temperature > 0`.

---

## ✅ Chapter 1 Summary

| Concept | What It Means |
|---------|---------------|
| Language Model | Predicts next token using learned patterns |
| Self-supervised learning | Trains on unlabeled text (no human labels needed) |
| Parameters | Numbers the model adjusts to learn — more = more capacity |
| Temperature | Controls randomness of generation — 0 = deterministic |
| Logits → Probabilities | Raw model scores converted to a probability distribution via softmax |

## 🔮 Up Next: Chapter 2 — Tokens and Embeddings

How does a model actually "see" text? Words are not fed in as strings — they are converted to **numbers** first. In Chapter 2, we explore **tokenization** and **embeddings** — the numerical representation that makes it all work.